# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [324]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display
import gradio as gr
import ollama
import textwrap

In [325]:
def chat_gpt(prompts):
    # gpt4o-miniの場合、tool_calls。他モデルは function_call
    CALL_FINISH_REASONS = {"function_call", "tool_calls"}
    stream = openai.chat.completions.create(model='gpt-4o-mini',
                                            messages=prompts,
                                            stream=True,
                                            tools=tools)
    # バッファ：id → { name, args_str }
    buffers = {} 
    current_id = None
    func_name = ""
    result = ""


    for chunk in stream:
        delta_calls = chunk.choices[0].delta.tool_calls
        if delta_calls is not None:
            for call in delta_calls:
                # 1) 新しい呼び出し開始: id, name が返ってくる
                if call.id and call.function.name:
                    current_id = call.id
                    func_name = call.function.name
                    buffers[current_id] = {
                        "name": func_name,
                        "args_str": ""
                    }
                # 2) arguments断片を累積
                #    新チャンクに id=None,name=None の場合は current_id を使う
                if call.function.arguments is not None and current_id:
                    buffers[current_id]["args_str"] += call.function.arguments
                    
        if chunk.choices[0].finish_reason in CALL_FINISH_REASONS:
            assistant_msg = {
                "role": "assistant",
                "content": None,
                "tool_calls": [
                    {
                        "id": current_id,
                        "function": {
                            "name": func_name,
                            "arguments": buffers[current_id]["args_str"]
                        },
                        "type": "function"
                    }
                ]
            }
            prompts.append(assistant_msg)

            responce = handle_tool_call(buffers)
            prompts.append(responce)
            print(tool_calls_accumulated)
            print(assistant_msg)
            yield from chat_gpt(prompts)
        else:
            result += chunk.choices[0].delta.content or ""
            yield result

In [326]:
def chat_ollama(prompts):
    stream = ollama.chat(model="gemma3:4b-it-qat",
                         messages=prompts,
                         stream=True)

    result = ""
    for chunk in stream:
        result += chunk['message']['content'] or ""
        yield result

In [327]:
def chat_model_handller(message, history):
    global CURRENT_MODEL
    system_message = """
    ## 役割
    あなたは航空会社のユーザーアシスタントです。

    ## 目的
    あなたの目的はユーザーが求める最適な情報にたどり着けるよう、航空会社のアシスタントとしてユーザーを支援することです。

    ## ルール
    以下のルールは必ず守ること。
    - 日本語で回答すること。
    - マークダウン形式で回答すること。
    """

    prompts = [{
        "role": "system",
        "content": system_message
    }] + history + [{
        "role": "user",
        "content": message
    }]

    if CURRENT_MODEL == "GPT":
        result = chat_gpt(prompts)
    elif CURRENT_MODEL == "OLLAMA":
        result = chat_ollama(prompts)
    else:
        print("undefined model")

    yield from result

In [328]:
def gradio_select_model(model_name):
    # ドロップダウンで選択されたモデル名を出力
    global CURRENT_MODEL
    print(f"選択されたモデル: {model_name}")
    CURRENT_MODEL = model_name

In [329]:
def get_ticket_price(destination_city):
    ticket_prices = {"ロンドン": "$799", "パリ": "$899", "東京": "$1400", "ベルリン": "$499"}
    return ticket_prices.get( destination_city, "Unknown" )    

In [330]:
def handle_tool_call(buffers):
    function_map = {"get_ticket_price": get_ticket_price}
    for id, content in buffers.items():
        func_name = (content["name"])
        args = json.loads(content["args_str"])

        func = function_map.get(func_name)

        return_value = func(**args)
        response = {
            "role": "tool",
            "content": json.dumps({
                "return_value": return_value
            }),
            "tool_call_id": id
        }

        return response

In [331]:
# AIに渡す関数情報
price_function = {
    "name":
    "get_ticket_price",
    "description":
    textwrap.dedent("""
        ## 関数が必要なタイミング
        ユーザーからチケットの価格を求められた際に関数が必要です。例えば、以下のような質問が来た時に必要です。
        - 東京行きのチケットの価格を教えてください。
        - ロンドン行きは何円でしょうか？
        
        ## 関数の概要
        ユーザーが指定した都市のチケットの価格を取得します。チケット価格が不明の都市の場合、Unknowを返します。                        
        """),
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "ユーザーがチケットの価格を求めている都市。",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [332]:
CURRENT_MODEL="OLLAMA"

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if not openai_api_key:
    print("APIキーが取得できませんでした。")
    pass

openai = OpenAI()

tools = [{"type": "function", "function": price_function}]

In [333]:
with gr.Blocks() as bl:
    with gr.Column(scale=1):
        gr.Markdown("### 設定")
        model_dropdown = gr.Dropdown(choices=["GPT", "OLLAMA"],
                                     label="モデルを選択",
                                     value="OLLAMA")
        model_dropdown.change(fn=gradio_select_model,
                              inputs=[model_dropdown],
                              outputs=[])
    with gr.Column(scale=9):
        gr.ChatInterface(fn=chat_model_handller,
                         type="messages",
                         theme=gr.themes.Soft(),
                         title="AIチャット")

bl.launch()

* Running on local URL:  http://127.0.0.1:7898
* To create a public link, set `share=True` in `launch()`.
